# Training Dynamics, Logging & Evaluation

Now let's examine the runtime behavior of GRPO training: What metrics do we log to TensorBoard/W&B, how do we spot training collapse early, and how do we evaluate Base vs. SFT vs. GRPO?

## 1. Core Metrics Dashboard

When running GRPO, you monitor four primary telemetry curves to judge system health:

### Key metrics

- Mean Reward ($\bar{r}$): tracks average reward across rollouts in the batch.
- KL Divergence ($D_{KL}$): tracks policy drift from the reference policy.
- Completion Length: tracks the average number of tokens generated per response.
- Advantage Standard Deviation: tracks reward variance within sampled groups.

### Visual dashboard

```text
Metric 1: Mean Reward (r_bar)              Metric 2: KL Divergence (D_KL)
    │     ▲                                    │     ▲
    │    /                                     │    / ─── (Healthy gentle rise)
    │   / ─── (Target: Steady ascent)          │   /
    └───┴───────────────►                      └───┴───────────────►
            Steps                                      Steps

Metric 3: Completion Length (Tokens)        Metric 4: Advantage Standard Deviation
    │     ▲                                    │     ▲
    │    / ─── (Spike = Length hacking)        │    ─── (Healthy: Non-zero variance)
    │   /                                      │    ___ (Zero = Dead batch collapse)
    └───┴───────────────►                      └───┴───────────────►
            Steps                                      Steps
```

### Metric reference

| Metric | What it measures | Healthy behavior | Failure signal |
|---|---|---|---|
| reward/mean | Average reward across all rollouts in the batch | Steady climb toward $1.0$ | Flatlining near $0.0$ or abrupt drop |
| objective/kl | Divergence between active policy $\pi_\theta$ and reference $\pi_{\text{ref}}$ | Small, stable, slowly increasing ($0.01 \to 0.2$) | Explosion ($> 1.0$) means catastrophic drift / model gibberish |
| completion_length | Average number of tokens generated per response | Moderate growth (developing reasoning steps) | Monotonic spike to max context length (length-bias reward gaming) |
| reward/std | Reward variance within the sampled groups ($G$) | Stable non-zero value ($0.2\text{–}0.5$) | Drops to $0.0$ (model mode collapses into identical outputs) |

## 2. Common GRPO Failure Modes & Engineering Fixes

### Failure Mode A: KL Explosion & Language Collapse

- Symptom: Mean reward increases briefly, then KL shoots through the roof, and the model starts outputting repetitive nonsense or garbled tokens that trigger regex edge cases.
- Root cause: The KL penalty coefficient ($\beta$) is too low, allowing the policy to drift into wild out-of-distribution token spaces.
- Fix: Increase the KL penalty coefficient $\beta$ (for example, $\beta = 0.04 \to 0.1$) or clip the advantage range.

### Failure Mode B: Length Gaming (Verbosity Drift)

- Symptom: Generation lengths hit the max sequence cap (for example, $2048$ tokens) on simple problems.
- Root cause: Reasoning tags or step-wise format rewards are giving micro-rewards that accumulate with longer text.
- Fix: Add a length-penalty term or reward strictly on terminal verification without rewarding text volume.

### Failure Mode C: Zero Advantage Starvation

- Symptom: Gradients become $\mathbf{0}$ across entire epochs, training completely stalls.
- Root cause: The prompts in the dataset are either too hard ($0\%$ pass rate) or too easy ($100\%$ pass rate), meaning all $G$ rollouts get identical rewards, making $A_i = 0$.
- Fix: Increase temperature ($T \approx 0.8\text{–}1.0$) to encourage broader exploration, increase group size $G$ (for example, $4 \to 8$), or curate a curriculum of medium-difficulty prompts where pass rate is $20\%\text{–}80\%$.